# Dicionário de Dados e Referências

**Dicionário de Dados de Notificação Geral - SINAN:** _http://portalsinan.saude.gov.br/images/documentos/Agravos/NINDIV/DIC_DADOS_NET_Not_Individual_rev.pdf_  
**Dicionario Ficha de Notificação Individual Geral - SINAN:** _http://portalsinan.saude.gov.br/images/documentos/Agravos/DRT%20Transtorno%20Mental/DIC_DADOS_DRT_TranstornosMentais_v5.pdf_  
**Classificação Brasileira de Ocupação:** _https://www.gov.br/trabalho-e-emprego/pt-br/assuntos/cbo/servicos/downloads/livro-1-portal-cbo.pdf_  
**Visualizador Web - TabNet:** _http://tabnet.datasus.gov.br/cgi/tabcgi.exe?sinannet/cnv/transmentalbr.def_

# Instalar as dependências para o R:

In [ ]:
# 1. Instalar dependências do sistema Linux necessárias para manipulação de dados e FTP
system("apt-get update && apt-get install -y libssl-dev libcurl4-openssl-dev libxml2-dev r-base-dev build-essential")

# 2. Instalar pacotes do R

options(repos = c(CRAN = "https://cloud.r-project.org"))

install.packages("remotes")
install.packages("rlang")
install.packages("tidyverse") # Inclui dplyr, ggplot2, etc.

# 3. Instalar o pacote microdatasus direto do repositório da Fiocruz (vai utilizar o CRAN setado anteriormente)
remotes::install_github("rfsaldanha/microdatasus", type = "source")

# 4. Carregar as bibliotecas
library("remotes")
library(microdatasus)
library(dplyr)

# As bibliotecas instaladas ficam em: /opt/conda/lib/R/library

Carregar/Verificar as bibliotecas:

In [8]:
library("remotes")
library(microdatasus)
library(dplyr)
#.libPaths()

# Baixando os dados referentes às bases SINAN para os anos de 2010 à 2025, com excessão de 2020 e 2021 (ausentes no SINAN).:

In [ ]:
anos_antigos <- "10 11 12 13 14 15 16 17 18 19"
anos_recentes <- "22 23 24 25"

system(paste("for ano in", anos_antigos, "; do wget -P ~/work ftp://ftp.datasus.gov.br/dissemin/publicos/SINAN/DADOS/FINAIS/MENTBR$ano.dbc; done"))
system(paste("for ano in", anos_recentes, "; do wget -P ~/work ftp://ftp.datasus.gov.br/dissemin/publicos/SINAN/DADOS/PRELIM/MENTBR$ano.dbc; done"))

#system("wget -P ~/work ftp://ftp.datasus.gov.br/dissemin/publicos/SINAN/DADOS/PRELIM/MENTBR24.dbc")

library(read.dbc)
dados <- read.dbc("~/work/MENTBR25.dbc")

head(dados)

Lendo e concatenando todos pacotes baixados referentes a cada ano:

In [ ]:
#library(read.dbc)
#library(dplyr)

# 1. Definir os anos que queremos (2010 a 2025, removendo 2020 e 2021)
anos <- 2010:2025
anos_filtrados <- anos[!anos %in% c(2020, 2021)]

# 2. Criar uma lista vazia para armazenar os dataframes temporariamente
lista_dados <- list()

# 3. O Laço de Repetição
for (ano in anos_filtrados) {

  # Extrai os dois últimos dígitos do ano (ex: 2015 vira "15")
  sufixo <- substr(as.character(ano), 3, 4)

  # Monta o caminho do arquivo dinamicamente
  caminho_arquivo <- paste0("~/work/MENTBR", sufixo, ".dbc")

  # Cria um nome de identificação para a lista (ex: "ano_15")
  nome_lista <- paste0("ano_", sufixo)

  # Mensagem no console para você acompanhar o progresso da leitura
  cat("Lendo arquivo:", caminho_arquivo, "\n")

  # Lê o arquivo e armazena na lista se ele existir
  if (file.exists(caminho_arquivo)) {
    lista_dados[[nome_lista]] <- read.dbc(caminho_arquivo)
  } else {
    warning(paste("Arquivo não encontrado:", caminho_arquivo))
  }
}

# 4. Junta todos os anos em um único grande dataframe
dados_completos <- bind_rows(lista_dados, .id = "FONTE_ANO")

# Remove a lista temporária para liberar memória RAM
rm(lista_dados)

# Salvando na memória secundária
saveRDS(dados_completos, file = "work/dados_completos_sinan.rds")

# dados <- readRDS("work/dados_completos_sinan.rds")

In [12]:
dados <- readRDS("~/work/dados_completos_sinan.rds")

In [13]:
head(dados)

,FONTE_ANO,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,⋯,ANO_NASC,NU_LOTE_I,DT_TRANSUS,DT_TRANSDM,DT_TRANSSM,DT_TRANSRM,DT_TRANSRS,DT_TRANSSE,NU_LOTE_V,NU_LOTE_H
,<chr>,<fct>,<fct>,<date>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,⋯,<fct>,<fct>,<date>,<date>,<date>,<date>,<date>,<date>,<fct>,<fct>
1,ano_10,2,F99,2010-01-27,201004,2010,28,280030,2056,5841399,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
2,ano_10,2,F99,2010-01-29,201004,2010,42,420910,1565,2436418,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
3,ano_10,2,F99,2010-01-05,201001,2010,35,355030,1331,2786664,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
4,ano_10,2,F99,2010-01-26,201004,2010,29,293330,1398,2550202,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
5,ano_10,2,F99,2010-01-15,201002,2010,29,292740,1380,2557894,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
6,ano_10,2,F99,2010-01-15,201002,2010,29,291480,1386,3432890,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [18]:
# Explorando a base de dados coadunada:
names(dados)
length(dados)
dim(dados)

[1] "FONTE_ANO"  "TP_NOT"     "ID_AGRAVO"  "DT_NOTIFIC" "SEM_NOT"   
 [6] "NU_ANO"     "SG_UF_NOT"  "ID_MUNICIP" "ID_REGIONA" "ID_UNIDADE"
[11] "DT_DIAG"    "SEM_DIAG"   "NU_IDADE_N" "CS_SEXO"    "CS_GESTANT"
[16] "CS_RACA"    "CS_ESCOL_N" "SG_UF"      "ID_MN_RESI" "ID_RG_RESI"
[21] "ID_PAIS"    "ID_OCUPA_N" "SIT_TRAB"   "NUTEMPO"    "TPTEMPO"   
[26] "CNAE"       "UF_EMP"     "MUN_EMP"    "TERCEIRIZA" "NUTEMPORIS"
[31] "TPTEMPORIS" "REGIME"     "DIAG_ESP"   "ALCOOL"     "PSICO_FARM"
[36] "DROGAS"     "FUMA"       "TEMPO_FUMA" "TP_TEMP_FU" "AFAST_DESG"
[41] "INDIVIDUAL" "MUDA_TRAB"  "NENHUM"     "COLETIVA"   "AFAST_TRAB"
[46] "CONDUTA"    "TRAB_DOE"   "CAPES"      "DT_OBITO"   "CAT"       
[51] "EVOLUCAO"   "CONDUT_DES" "DT_DIGITA"  "ANO_NASC"   "NU_LOTE_I" 
[56] "DT_TRANSUS" "DT_TRANSDM" "DT_TRANSSM" "DT_TRANSRM" "DT_TRANSRS"
[61] "DT_TRANSSE" "NU_LOTE_V"  "NU_LOTE_H"

[1] 63

[1] 26873    63

**Averiguando variedade das categorias de transtornos mentais para a variável de Diagnóstico Específico:**

In [15]:
dados$DIAG_ESP

[1] F419 F43  F333 F431 F99  F432 F432 F432 F412 F332 F430 F432 F432 F32 
   [15] F32  F32  F431 F322 F430 F333 F454 F32  F413 F431 F32  F38  F330 F332
   [29] F431 F410 F431 F431 F321 Z730 Z578 F329 F321 F43  F43  F29  <NA> F32 
   [43] F313 F41  F32  F321 F332 F412 F438 F432 F432 F323 F431 <NA> F409 F410
   [57] F331 F321 F432 F321 F333 F331 F431 F431 <NA> F431 F431 F43  F99  F411
   [71] F41  F432 F431 F99  F412 F332 F431 F432 <NA> F432 F322 F43  F431 F321
   [85] F43  F411 F43  F43  F32  F412 F32  F42  F431 F431 F431 F322 F32  F32 
   [99] F31  F431 F333 F430 F320 F41  F32  F32  I50  F321 F430 F209 F431 F322
  [113] F43  <NA> F320 F431 F321 F329 F322 F51  F431 F431 F329 F33  F438 F43 
  [127] F430 F43  F322 F431 F32  F32  F32  F431 T434 F32  F32  F32  F43  F70 
  [141] <NA> F339 F431 F32  <NA> F411 F40  F332 F420 F332 F412 F419 F321 F321
  [155] <NA> <NA> F43  F430 F43  F99  F07  F43  F20  F41  F512 <NA> F322 Z56 
  [169] <NA> F41  F99  F422 F322 F410 F43  F231 F432 F431 <NA> F430 F43  F432
  [183] F411 <NA> F32  F333 F320 F410 F43  F431 F321 F432 H903 F51  F32  F431
  [197] F431 F322 F431 F431 F430 F41  F322 F323 F322 F432 F321 F43  F432 G47 
  [211] F322 F409 Z565 F33  F40  F322 F48  F43  F438 Z565 F32  F32  F99  F411
  [225] F432 F348 F328 F40  F321 F322 F432 F431 F431 <NA> F411 F320 F41  F41 
  [239] F41  F41  F410 F32  F31  <NA> F32  F322 F431 F43  F33  F42  F431 F430
  [253] F99  F40  F431 F43  F332 F99  F99  F412 F431 F43  F322 F380 F41  F322
  [267] F321 F329 F32  F431 F431 F32  F431 F431 F431 F410 F431 F410 <NA> F078
  [281] F43  F410 F432 F41  F072 Z730 F411 F33  F439 F322 F430 F431 F40  F412
  [295] F43  F43  F438 F431 F438 F488 F43  S43  F99  F99  F99  F430 F431 F331
  [309] F322 F32  F43  F320 F320 F99  F32  F42  F329 F32  F411 F410 F419 F419
  [323] F412 F431 F322 F99  F32  <NA> <NA> F431 F412 F322 F33  F412 F431 F32 
  [337] F32  F412 F412 F431 F412 F321 <NA> F438 F323 F438 F41  F331 F412 F322
  [351] F33  F431 F431 F32  F430 F431 <NA> F430 F431 F431 F431 F431 F431 F99 
  [365] F322 F32  F32  F321 F431 F43  F32  F322 <NA> F432 F432 F431 <NA> <NA>
  [379] F32  F99  F410 F32  <NA> F512 F432 F33  F322 <NA> F410 F412 F432 F432
  [393] F41  F32  F32  F43  F431 F43  F43  F412 F412 F322 F32  F99  F431 F43 
  [407] F233 F431 F431 F430 F401 F419 <NA> F322 F43  F45  F412 F32  F430 F32 
  [421] <NA> F431 F43  F41  F430 F430 <NA> F412 F431 <NA> <NA> Z730 F431 <NA>
  [435] F300 F321 F43  <NA> <NA> F43  F41  F320 F410 <NA> <NA> F322 F402 <NA>
  [449] F32  <NA> Z566 F20  F432 F419 <NA> F32  F454 F430 F431 F431 F412 F410
  [463] F431 F431 F430 F431 F431 F432 F431 F431 F430 F300 F32  F32  T431 F431
  [477] F438 Z73  F431 F431 F32  Z566 F432 F332 F32  F43  <NA> <NA> F439 F430
  [491] F430 F430 F430 F430 F430 F430 F430 F430 F430 F430 F430 F430 F430 F430
  [505] F430 F99  F32  F411 F43  F432 F322 F439 F43  F41  I10  <NA> F322 F432
  [519] F32  F430 F32  F03  F30  F430 F322 <NA> F43  F43  F412 F20  F41  F32 
  [533] F32  F321 F322 F43  Z730 Z565 F431 <NA> F43  F430 F430 F431 F321 <NA>
  [547] F320 F412 F40  Z730 F32  F44  F448 F411 F31  F32  F410 F322 F43  F411
  [561] F32  F402 Z609 F332 F431 F322 F322 F43  F431 F431 F32  F99  F99  F99 
  [575] F99  F99  F99  Z730 F410 F330 F431 <NA> <NA> F43  <NA> F431 F43  F439
  [589] F99  F232 F430 F431 F333 F321 F32  F41  F321 F320 F411 <NA> F32  Z730
  [603] Z730 F322 F322 F318 F32  F321 F43  F32  F410 F320 F99  F99  F431 F432
  [617] F32  F432 F431 <NA> F438 F431 F422 F321 F323 F431 F410 F431 F32  F431
  [631] F43  F432 <NA> F063 F410 F431 F432 F411 <NA> F322 F412 F41  F43  F431
  [645] F432 Z565 F321 F43  F43  F99  F32  F32  F41  F410 <NA> F33  F410 F43 
  [659] F32  F43  F438 F412 F450 <NA> F431 F412 F431 F431 F431 F99  F412 F40 
  [673] <NA> <NA> F43  F412 F431 F432 F431 F43  F322 F450 F42  F431 F99  <NA>
  [687] F41  F43  F430 F31  F321 F43  F488 F431 F431 F432 F331 F412 F43  F99 
  [701] F99  F99  F99  <NA> F431 F329 <NA> F332 F99  F99  F99  F43  F430 F419
  [715] F432 F430 F431 <NA

**Averiguando integridade e disponibilidade dos estados para a variável 'Estado de Notificação':**

In [22]:
dados$SG_UF_NOT

[1] 28 42 35 29 29 29 26 26 29 24 24 35 35 50 50 50 50 50 35 50 50 50 50 50
   [25] 35 35 26 28 29 26 35 35 35 35 42 35 35 29 31 29 17 23 23 25 35 50 29 26
   [49] 35 35 35 35 41 42 42 35 35 35 35 35 35 35 35 35 32 35 35 50 28 28 29 35
   [73] 35 26 24 24 35 14 42 35 35 31 35 35 50 28 28 28 28 28 28 28 28 35 35 35
   [97] 31 31 31 29 24 24 17 31 31 31 35 35 35 31 35 31 31 31 35 35 35 35 35 35
  [121] 35 35 35 50 42 28 28 35 28 28 31 31 31 29 17 17 35 35 31 29 35 31 31 31
  [145] 31 17 23 31 17 35 35 35 35 35 13 50 35 29 35 35 35 35 35 35 35 17 28 28
  [169] 35 31 42 42 42 35 35 35 35 35 35 31 31 31 31 35 31 35 29 35 35 35 35 35
  [193] 29 35 35 41 41 29 35 35 28 31 31 31 31 35 35 35 35 35 35 31 35 35 29 29
  [217] 24 24 42 35 31 31 35 23 23 42 35 35 35 35 35 35 35 42 23 23 31 31 31 31
  [241] 35 35 29 35 43 17 29 35 31 28 29 29 17 31 35 35 41 35 35 35 31 50 24 35
  [265] 35 35 35 35 35 31 31 31 31 31 31 31 31 31 41 29 50 35 35 31 31 41 28 17
  [289] 35 35 35 29 35 41 42 42 42 35 42 42 50 50 28 28 28 29 31 35 35 35 35 35
  [313] 31 35 31 23 35 35 35 33 23 23 29 41 35 25 31 43 31 31 35 35 35 31 31 35
  [337] 35 35 35 35 35 41 29 42 42 42 17 17 28 29 41 35 35 35 42 31 35 35 31 31
  [361] 31 31 31 35 43 31 31 29 31 29 35 35 43 50 50 42 35 33 35 17 17 35 31 29
  [385] 29 17 35 41 35 35 23 41 35 35 35 35 35 35 35 35 35 35 35 23 50 42 35 29
  [409] 29 25 29 23 50 29 35 14 17 31 41 35 26 31 23 35 29 29 23 29 42 23 51 51
  [433] 31 35 35 35 42 26 26 42 32 32 32 15 23 35 17 17 35 23 23 35 26 35 17 35
  [457] 17 35 35 29 28 35 29 29 29 29 35 24 35 35 35 29 29 29 28 50 35 35 24 35
  [481] 35 35 35 26 35 42 17 31 29 41 41 41 41 41 41 41 41 41 41 41 41 41 41 41
  [505] 41 35 35 17 31 35 42 28 42 29 33 41 29 29 23 41 35 29 29 29 26 23 42 50
  [529] 35 13 31 29 29 13 17 42 11 23 27 35 42 31 31 29 11 26 29 26 31 29 43 17
  [553] 43 24 50 41 24 50 42 24 24 35 31 35 29 29 29 50 29 43 35 33 33 33 33 33
  [577] 33 26 26 26 26 26 26 26 26 26 42 31 33 35 29 42 29 35 35 35 35 35 35 35
  [601] 23 35 35 35 35 35 35 35 35 35 35 35 35 35 35 35 23 35 26 35 43 31 17 35
  [625] 28 43 35 31 28 26 26 26 31 17 35 31 35 35 35 29 35 35 50 31 35 35 35 35
  [649] 23 28 52 52 52 35 35 23 28 42 23 42 31 35 26 26 26 26 35 42 29 29 17 35
  [673] 41 41 28 26 42 41 31 35 35 26 23 31 50 31 28 42 43 21 31 35 31 29 31 35
  [697] 25 31 13 13 13 13 13 13 41 35 35 35 13 13 13 42 13 51 29 29 29 41 31 31
  [721] 35 31 31 13 29 35 31 42 31 31 31 41 26 41 26 31 29 35 35 28 17 35 13 13
  [745] 35 52 50 23 31 31 31 31 31 31 31 31 31 13 52 13 13 31 23 26 35 42 35 42
  [769] 35 31 31 51 29 26 42 23 50 35 35 52 35 35 13 13 31 35 35 24 13 35 43 35
  [793] 35 31 31 31 35 35 35 35 35 35 35 35 35 42 43 43 43 35 26 35 31 31 43 43
  [817] 26 42 50 13 31 52 43 29 29 29 31 43 31 31 31 31 35 35 35 31 41 41 41 31
  [841] 35 29 31 50 35 35 31 31 35 21 33 35 26 31 31 31 31 31 35 52 35 26 31 31
  [865] 29 50 31 35 42 31 35 23 35 42 26 28 24 26 50 35 31 35 31 28 31 29 29 24
  [889] 26 41 24 35 33 31 35 42 31 35 31 31 42 35 11 35 31 35 31 35 52 26 25 35
  [913] 32 35 35 28 35 23 29 35 11 31 23 31 32 21 21 21 21 21 21 21 21 21 21 32
  [937] 32 42 31 24 31 35 31 31 29 35 35 35 29 35 35 35 28 43 31 31 29 11 11 11
  [961] 31 31 33 35 28 26 11 35 41 35 35 26 35 35 35 35 23 11 35 31 31 31 31 31
  [985] 31 26 35 11 50 31 51 32 24 26 50 50 11 11 11 31 31 31 31 31 31 42 29 29
 [1009] 26 31 31 33 42 31 35 35 26 31 31 31 31 31 52 24 29 29 26 26 29 31 35 35
 [1033] 29 13 41 35 13 13 31 41 17 24 24 32 31 31 42 33 35 35 35 35 35 43 23 26
 [1057] 31 35 43 43 32 29 31 35 35 35 35 17 35 35 35 35 29 35 35 35 35 35 35 24
 [1081] 31 23 50 35 35 35 26 26 35 26 35 35 35 35 35 35 35 35 35 41 26 35 31 26
 [1105] 42 35 35 31 31 31 31 26 26 31 31 31 31 29 26 35 50 31 28 32 32 32 35 42
 [1129] 29 50 35 32 32 42 29 31 35 32 29 29 35 29 29 41 41 31 31 31 31 42 42 29
 [1153] 35 35 26 41 41 31 31 31 42 41 35 35 28 35 29 35 29 24 41 41 31 31 31 23
 [1177] 35 35 35 23 35 24 31 31 31 31 33 42 31 26 35 26 35 31 31 31 35 50 29 31
 [12

_A base representa os casos notificados formalmente ao sistema de vigilância nacional, sabendo que o volume real de indivíduos com Burnout tratados por psicólogos e psiquiatras particulares/de convênio médico está altamente sub-representado nesses números._

In [26]:
# 1. Vetor nomeado para servir de "dicionário" de tradução
tradutor_uf <- c(
  "11" = "RO", "12" = "AC", "13" = "AM", "14" = "RR", "15" = "PA", "16" = "AP", "17" = "TO",
  "21" = "MA", "22" = "PI", "23" = "CE", "24" = "RN", "25" = "PB", "26" = "PE", "27" = "AL",
  "28" = "SE", "29" = "BA", "31" = "MG", "32" = "ES", "33" = "RJ", "35" = "SP", "41" = "PR",
  "42" = "SC", "43" = "RS", "50" = "MS", "51" = "MT", "52" = "GO", "53" = "DF"
)

In [32]:
tabela_Burnout <- dados %>%
  mutate(
    UF_Codigo = trimws(as.character(SG_UF)),
    UF_SG = tradutor_uf[UF_Codigo],
  ) %>%
  filter(
    DIAG_ESP   == "Z730"
  ) %>%
  group_by(UF_SG) %>%
  summarise(Casos = n(), .groups = "drop") %>%
  arrange(UF_SG)

head(tabela_Burnout, n = Inf)

UF_SG,Casos
<chr>,<int>
AC,4
AL,8
AM,2
BA,238
CE,64
DF,6
ES,6
GO,4
MA,7


Podemos verificar que não há registros referentes nem à Piauí (**PI**) nem à  Amapá (**AP**) para a base de dados utilizada.

# Selecionar as variáveis de interesse e tratar a variável de 'Estado De Notificação':

In [37]:
tabela_Burnout <- dados %>%
  select(FONTE_ANO, DIAG_ESP, SG_UF, SIT_TRAB, ID_OCUPA_N, CNAE, UF_EMP,
  TERCEIRIZA, NUTEMPORIS, TPTEMPO, AFAST_DESG, AFAST_TRAB, CAPES, EVOLUCAO) %>%
  mutate(
    UF_Codigo = trimws(as.character(SG_UF)),
    UF_SG = tradutor_uf[UF_Codigo],
  ) %>% select(-SG_UF, -UF_Codigo) %>%
    relocate(UF_SG, .after = DIAG_ESP) %>%
  filter(
    DIAG_ESP   == "Z730"
  )
# Exibe o resultado final com os dados tratados mediante dicionário:
#print(tabela_Burnout, n = Inf)

#saveRDS(tabela_Burnout, file = "work/dados_tratados_sinan.rds")
#tabela_Burnout <- readRDS('work/dados_tratados_sinan.rds')

head(tabela_Burnout)

,FONTE_ANO,DIAG_ESP,UF_SG,SIT_TRAB,ID_OCUPA_N,CNAE,UF_EMP,TERCEIRIZA,NUTEMPORIS,TPTEMPO,AFAST_DESG,AFAST_TRAB,CAPES,EVOLUCAO
,<chr>,<fct>,<chr>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>
1,ano_10,Z730,SP,01,513425,NA,35,2,5,3,1,1,1,3
2,ano_10,Z730,PR,01,239405,NA,41,2,4,NA,2,1,1,3
3,ano_11,Z730,MT,04,411010,NA,51,2,20,4,1,NA,1,NA
4,ano_11,Z730,RO,04,223405,NA,11,2,5,4,2,2,1,3
5,ano_11,Z730,BA,07,410105,NA,NA,2,0,NA,1,1,1,3
6,ano_11,Z730,PE,04,331105,05126,26,2,20,4,1,NA,1,4
